# Notebook 3: AI Agents and Agent-to-Agent (A2A) Evaluation

This notebook demonstrates AI agents for portfolio optimization with **automatic Langfuse tracing** via `@observe()` decorator.

## A2A Protocol Overview

The A2A (Agent-to-Agent) protocol follows the **Green/Purple agent pattern**:

- **Purple Agent** (Portfolio) - The agent being evaluated, handles investor requests
- **Green Agent** (Evaluator) - Defines tasks, queries the Purple agent, and produces assessments

### Protocol Flow
1. **Round 1**: Green sends request → Purple responds with portfolio recommendation
2. **Round 2**: Green uses tools (RAG knowledge base + web search) to gather context
3. **Final Assessment**: Green produces scores and feedback

**Key**: All communication is logged as `A2AMessage` objects for full traceability.

## Setup

In [1]:
import os
import json
import warnings
from typing import Dict, Any, List, Optional
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
load_dotenv(dotenv_path='../.env')

# Configure Gemini API
import google.generativeai as genai
gemini_api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if gemini_api_key:
    genai.configure(api_key=gemini_api_key)
    os.environ["GOOGLE_API_KEY"] = gemini_api_key

from langchain.agents import AgentExecutor

# Import agents - A2A protocol with Green/Purple agents
from agents import (
    # A2A Protocol agents
    create_a2a_purple_agent,  # Portfolio agent (being evaluated)
    create_a2a_green_agent,   # Evaluator agent (RAG + web search)
    run_a2a_evaluation,
    A2AEvaluation,
    A2AMessage,
    # Utilities
    create_rag_knowledge_base,
    run_portfolio_agent,
    PORTFOLIO_TOOLS,
    flush_langfuse,
    LLMProvider
)
from portfolio_optimizer import UNIVERSE_DEFINITIONS

# Load data from JSON files
with open('scenarios.json', 'r') as f:
    SCENARIOS = json.load(f)
with open('evaluation_dataset.json', 'r') as f:
    EVAL_DATA = json.load(f)

# Default provider for this notebook (Gemini)
DEFAULT_PROVIDER = LLMProvider.GEMINI

warnings.filterwarnings('ignore')
print(f"Gemini API configured: {'Yes' if gemini_api_key else 'No'}")
print(f"Default LLM Provider: {DEFAULT_PROVIDER.value}")
print(f"Loaded {len(SCENARIOS['personas'])} personas, {len(EVAL_DATA['items'])} eval items")
print("\nA2A Protocol: Green (Evaluator) ↔ Purple (Portfolio)")
print("Setup complete!")

Gemini API configured: Yes
Default LLM Provider: gemini
Loaded 3 personas, 20 eval items

A2A Protocol: Green (Evaluator) ↔ Purple (Portfolio)
Setup complete!


## 1. Purple Agent (Portfolio Optimizer)

The **Purple Agent** is the agent being evaluated. It has four portfolio tools:
- `get_available_universes` - List available asset universes
- `optimize_portfolio_tool` - Run MVO optimization
- `run_hrp_optimization` - Run HRP optimization
- `compare_portfolios` - Compare optimization methods

In the A2A protocol, Purple agents are "competitors" that attempt to excel at tasks defined by Green agents.

In [2]:
# Show available tools
print(f"Portfolio Agent Tools ({len(PORTFOLIO_TOOLS)}):")
for tool in PORTFOLIO_TOOLS:
    print(f"  - {tool.name}: {tool.description[:50]}...")

Portfolio Agent Tools (4):
  - get_available_universes: Get available asset universes for portfolio constr...
  - optimize_portfolio_tool: Optimize a portfolio using Mean-Variance Optimizat...
  - run_hrp_optimization: Run Hierarchical Risk Parity optimization....
  - compare_portfolios: Compare different optimization approaches for the ...


In [3]:
# Create Purple Agent (Portfolio Optimizer)
purple_agent = create_a2a_purple_agent(provider=DEFAULT_PROVIDER)
print(f"Purple Agent (Portfolio) created with {DEFAULT_PROVIDER.value}!")

Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'default' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring


Purple Agent (Portfolio) created with gemini!


In [4]:
# Test Purple Agent with Elena's persona
elena = SCENARIOS['personas']['elena_balanced']
print(f"Testing Purple Agent with {elena['name']}'s persona...\n")

# run_portfolio_agent uses @observe() - automatic tracing
result = run_portfolio_agent(purple_agent, elena['narrative'], session_id="notebook3_demo")

print("\n" + "="*60)
print("PURPLE AGENT RESPONSE (truncated):")
print(result["output"][:500] + "..." if len(result["output"]) > 500 else result["output"])
print("\n" + "="*60)
print(f"Tools used: {len(result.get('intermediate_steps', []))} tool calls")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Testing Purple Agent with Elena Rodriguez's persona...



I0000 00:00:1768482573.189029 29647652 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers



Invoking: `optimize_portfolio_tool` with `{'max_position': 0.15, 'universe': 'global', 'optimization_target': 'max_sharpe'}`
responded: Okay, Elena. Given your investment goals, time horizon, and risk tolerance, let's construct a portfolio for your children's education fund.

**1. Understand your goals:**

*   **Goal:** Build a portfolio for your children's education fund.
*   **Time Horizon:** Approximately 18 years (long-term).
*   **Risk Tolerance:** Moderate.
*   **Investment Amount:** $150,000.
*   **Preferences:** Global diversification, reasonable risk-adjusted returns, and no single position exceeding 15% of the portfolio.

**2. Select an appropriate universe:**

Given your preference for global diversification, I'll use the "global" universe.

**3. Choose the right optimization method:**

Considering your moderate risk tolerance and desire for reasonable risk-adjusted returns, I'll start with Mean-Variance Optimization with the goal of maximizing the Sharpe Ratio. This approa

## 2. Green Agent (Evaluator)

The **Green Agent** is the evaluator with two tools:
- `search_knowledge_base` - RAG retrieval from evaluation dataset
- `web_search` - DuckDuckGo web search for current information

In the A2A protocol, Green agents define tasks, environments, and scoring criteria. They query the Purple agent and produce assessments.

In [5]:
# Create RAG knowledge base for the Green Agent
retriever = create_rag_knowledge_base(EVAL_DATA)
print(f"RAG knowledge base created for Green Agent")
print(f"Personas included: {', '.join([p['persona'] for p in EVAL_DATA['rag_knowledge']])}")

RAG knowledge base created for Green Agent
Personas included: Sarah Chen, Marcus Johnson, Elena Rodriguez


In [6]:
# Create Green Agent (Evaluator) with RAG + Web Search
green_agent = create_a2a_green_agent(retriever, provider=DEFAULT_PROVIDER)
print(f"Green Agent (Evaluator) created with {DEFAULT_PROVIDER.value}")
print("Tools: search_knowledge_base (RAG), web_search (DuckDuckGo)")

Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring


Green Agent (Evaluator) created with gemini
Tools: search_knowledge_base (RAG), web_search (DuckDuckGo)


## 3. Agent-to-Agent (A2A) Evaluation Protocol

The A2A protocol orchestrates **iterative** communication between Green and Purple agents:

```
A2A Protocol Flow (max_rounds=3)
────────────────────────────────
Round 1: Initial Request
├── GREEN → PURPLE: "Please provide a portfolio recommendation for..."
└── PURPLE → GREEN: Portfolio recommendation response

Round 2: Follow-up Query
├── GREEN asks follow-up question (probing deeper)
└── PURPLE responds with more details

Round 3: Additional Query  
├── GREEN asks another question (clarification/justification)
└── PURPLE responds

Final Assessment
└── GREEN produces scores (1-10) and detailed feedback
```

The Green agent uses its tools during rounds to gather context:
- `search_knowledge_base` - RAG for best practices
- `web_search` - DuckDuckGo for current market info

**Langfuse Trace Structure:**
```
a2a_evaluation (trace)
├── Round 1: Purple Agent chain
├── Round 2: Green Agent query → Purple Agent response
├── Round 3: Green Agent query → Purple Agent response
└── Final: Green Agent assessment
```

In [7]:
# Run A2A evaluation: Green Agent evaluates Purple Agent
elena = SCENARIOS['personas']['elena_balanced']

print(f"A2A EVALUATION: Evaluating portfolio recommendation for {elena['name']}\n")
print("Protocol: GREEN (Evaluator) ↔ PURPLE (Portfolio)")
print("Rounds: 3 (initial + 2 follow-ups)")
print("="*60)

# Run the A2A protocol with 3 rounds of communication
a2a_result = run_a2a_evaluation(
    task_description=elena['narrative'],
    portfolio_agent=purple_agent,    # Purple Agent (being evaluated)
    evaluator_agent=green_agent,     # Green Agent (evaluator)
    session_id="notebook3_a2a",
    max_rounds=3  # 3 rounds: initial + 2 follow-up queries
)

# Display results
print("\n" + "="*60)
print("A2A EVALUATION RESULTS")
print("="*60)
print(f"\nTotal messages exchanged: {len(a2a_result.conversation)}")
print(f"Overall Score: {a2a_result.overall_score:.1f}/10")

print("\nDimension Scores:")
for dim, score in a2a_result.scores.items():
    print(f"  {dim}: {score:.1f}/10")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


A2A EVALUATION: Evaluating portfolio recommendation for Elena Rodriguez

Protocol: GREEN (Evaluator) ↔ PURPLE (Portfolio)
Rounds: 3 (initial + 2 follow-ups)

A2A PROTOCOL - ROUND 1: Initial Request

[GREEN → PURPLE] Please provide a portfolio recommendation for this investor: I'm Elena, 38 years old, and I want to build a portfolio for my children's education fund. My kids are 2 and 5 years old, so I have about 1...

Invoking: `get_available_universes` with `{}`
responded: Okay Elena, I can help you construct a portfolio for your children's education fund. Given your 18-year time horizon and moderate risk tolerance, a globally diversified portfolio using Mean-Variance Optimization seems appropriate. I will set a maximum position size of 15% as you requested.

First, let's check the available universes:


{
  "us_large_cap": [
    "AAPL",
    "MSFT",
    "GOOG",
    "AMZN",
    "META",
    "NVDA",
    "TSLA",
    "BRK-B",
    "JPM",
    "JNJ",
    "V",
    "UNH",
    "HD",
    "PG",
   

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Okay, Elena, let's build a portfolio for your children's education fund. Given your 18-year time horizon, moderate risk tolerance, and desire for global diversification, I recommend using the "global_diversified" universe. I will use Mean-Variance Optimization with the goal of maximizing the Sharpe Ratio, which balances risk and return. I will also set a maximum position size of 15% to ensure diversification.

Here's the portfolio I've created:

*   **Universe:** global\_diversified
*   **Optimization Target:** max\_sharpe
*   **Maximum Position:** 15%

**Portfolio Composition:**

*   AGG: 15.00%
*   BND: 4.19%
*   DBC: 15.00%
*   GLD: 15.00%
*   GOVT: 15.00%
*   IEF: 5.81%
*   SPY: 15.00%
*   VTI: 15.00%

**Backtest Results:**

*   **Sharpe Ratio:** 0.631
*   **Expected Return:** 7.84%
*   **Volatility:** 9.25%
*   **Max Drawdown:** -15.4%

This portfolio provides broad diversification across global equities (SPY, VTI), developed international equities (EFA), emerging market equities 

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


...

Okay, Elena, those are excellent questions. Let's break down the reasoning behind the portfolio construction:

**1. Heavy Weighting Explanation:**

The optimization process aims to maximize the Sharpe Ratio, which means finding the portfolio with the highest risk-adjusted return. The assets that ended up with the maximum 15% allocation (AGG, DBC, GLD, GOVT, SPY, and VTI) likely contribute the most to the portfolio's Sharpe Ratio within the given constraints. This could be due to a combination of factors:

*   **High Expected Returns:** These assets may have higher expected returns compared to others in the universe.
*   **Low Correlation:** They might have relatively low correlations with each other, which helps to diversify the portfolio and reduce overall volatility.
*   **Risk-Adjusted Performance:** Even if some assets have higher volatility, their return potential might be high enough to justify a larger allocation from a Sharpe Ratio perspective.

The lower allocations to assets 

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Okay, I appreciate the detailed explanation. It clarifies the reasoning behind the portfolio construction and the limitations of the current API. However, I still have some concerns about the risk assessment and the choice of optimization method, especially given the investor's moderate risk tolerance and long time horizon.

Here's my follow-up question for the Purple Agent:

"Thank you for the detailed response. I understand the rationale for maximizing the Sharpe Ratio and the constraints of the API. However, I'm still not fully convinced that the 'max_sharpe' optimization adequately addresses Elena's moderate risk tolerance, especially considering the portfolio's maximum drawdown of -15.4%.

1.  Could you provide more details on how the expected returns and volatilities for each asset in the 'global_diversified' universe were estimated? What historical data or forecasting methods were used? The accuracy of these estimates significantly impacts the Sharpe Ratio and the resulting port

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Those are excellent points, Elena. You're right to be concerned about the limitations of relying solely on the Sharpe Ratio and the sensitivity of Mean-Variance Optimization to input parameters. Let's address your questions:

1.  **Estimation of Expected Returns and Volatilities:**

I do not have access to the specific details on how the expected returns and volatilities for each asset in the 'global\_diversified' universe were estimated. This information is proprietary to the provider of the API. However, in general, these estimates are typically derived from historical data, potentially incorporating macroeconomic factors and analyst forecasts. The historical data usually spans several years to capture different market conditions. Different forecasting methods include:

*   **Historical Averages:** Using the average historical returns and volatilities over a specific period.
*   **Time Series Models:** Applying statistical models like ARIMA or GARCH to forecast future returns and vol

In [8]:
# Display A2A conversation history - shows iterative communication
print("A2A CONVERSATION HISTORY")
print("="*60)
print(f"\nTotal messages: {len(a2a_result.conversation)}")
print("Message flow: GREEN → PURPLE → GREEN → PURPLE → GREEN → PURPLE → GREEN (assessment)")

for i, msg in enumerate(a2a_result.conversation):
    sender_label = "GREEN (Evaluator)" if msg.sender == "green" else "PURPLE (Portfolio)"
    print(f"\n[{i+1}] {sender_label} - {msg.message_type.upper()}")
    print("-" * 40)
    # Truncate long messages for display
    content = msg.content[:500] + "..." if len(msg.content) > 500 else msg.content
    print(content)

A2A CONVERSATION HISTORY

Total messages: 7
Message flow: GREEN → PURPLE → GREEN → PURPLE → GREEN → PURPLE → GREEN (assessment)

[1] GREEN (Evaluator) - REQUEST
----------------------------------------
Please provide a portfolio recommendation for this investor: I'm Elena, 38 years old, and I want to build a portfolio for my children's education fund. My kids are 2 and 5 years old, so I have about 18 years. I have $150,000 to invest with moderate risk tolerance. I'd like global diversification and reasonable risk-adjusted returns. No single position should exceed 15% of the portfolio.

[2] PURPLE (Portfolio) - RESPONSE
----------------------------------------
Okay, Elena, let's build a portfolio for your children's education fund. Given your 18-year time horizon, moderate risk tolerance, and desire for global diversification, I recommend using the "global_diversified" universe. I will use Mean-Variance Optimization with the goal of maximizing the Sharpe Ratio, which balances risk and r

## 4. Batch A2A Evaluation

Run A2A evaluation on multiple scenarios. Each scenario creates two agent traces.

In [9]:
# Load complex scenarios
COMPLEX_SCENARIOS = SCENARIOS['complex_scenarios']
print(f"Complex scenarios ({len(COMPLEX_SCENARIOS)}):")
for s in COMPLEX_SCENARIOS:
    print(f"  - {s['name']}")

Complex scenarios (3):
  - Multi-objective comparison for Elena
  - Position limit analysis for Marcus
  - Risk strategy for Sarah


In [10]:
def evaluate_complex_scenarios(
    scenarios: List[Dict],
    purple_agent: AgentExecutor,
    green_agent: AgentExecutor,
    max_scenarios: int = 5
) -> List[A2AEvaluation]:
    """Run A2A evaluation on multiple scenarios.
    
    Each scenario runs the full A2A protocol:
    - Round 1: Green → Purple → Green
    - Round 2: Green uses tools for context
    - Final: Green produces assessment
    """
    results = []
    
    for i, scenario in enumerate(scenarios[:max_scenarios]):
        print(f"\n{'#'*60}")
        print(f"SCENARIO {i+1}: {scenario['name']}")
        print(f"{'#'*60}")
        
        # Run A2A evaluation
        eval_result = run_a2a_evaluation(
            task_description=scenario["description"],
            portfolio_agent=purple_agent,
            evaluator_agent=green_agent,
            session_id=f"notebook3_complex_{i}"
        )
        results.append(eval_result)
        
        print(f"\nScore: {eval_result.overall_score:.1f}/10")
        print(f"Messages exchanged: {len(eval_result.conversation)}")
    
    return results

print("Evaluating complex scenarios with A2A protocol...\n")
complex_results = evaluate_complex_scenarios(
    COMPLEX_SCENARIOS,
    purple_agent,
    green_agent,
    max_scenarios=1
)

print("\n" + "="*60)
print("Complex scenario evaluation complete!")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Evaluating complex scenarios with A2A protocol...


############################################################
SCENARIO 1: Multi-objective comparison for Elena
############################################################

A2A PROTOCOL - ROUND 1: Initial Request

[GREEN → PURPLE] Please provide a portfolio recommendation for this investor: Elena Rodriguez wants to compare three different approaches for her children's education fund ($150,000, 18-year horizon, moderate risk): 1...

Invoking: `get_available_universes` with `{}`
responded: Okay, Elena is planning for her children's education fund with a moderate risk tolerance and an 18-year time horizon. She wants to explore three portfolio optimization approaches: minimum volatility, maximum Sharpe ratio, and Hierarchical Risk Parity (HRP). Let's analyze each approach and provide a recommendation.

First, I need to know what asset universes are available.


{
  "us_large_cap": [
    "AAPL",
    "MSFT",
    "GOOG",
    "AMZN",
    "META

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Okay, Elena, let's analyze these three portfolio options for your children's education fund.

**1. Minimum Volatility (Conservative Assets):**

*   **Universe:** Conservative (BND, AGG, TLT, IEF, GOVT, LQD, MBB, VMBS)
*   **Optimization Target:** min\_volatility
*   **Maximum Position:** 15%
*   **Expected Return:** 0.72%
*   **Volatility:** 6.4%
*   **Sharpe Ratio:** -0.201
*   **Max Drawdown:** -19.4%
*   **Weights:** AGG: 0.15, BND: 0.15, GOVT: 0.15, IEF: 0.15, LQD: 0.1, MBB: 0.15, VMBS: 0.15

This portfolio prioritizes minimizing risk. It has the lowest volatility among the three options. However, the expected return is also very low, and the Sharpe ratio is negative, indicating poor risk-adjusted return. The max drawdown is also significant.

**2. Maximum Sharpe Ratio (Global Diversified Assets):**

*   **Universe:** Global Diversified (SPY, EFA, EEM, VWO, TLT, GLD, VNQ, LQD, HYG, DBC, IEF, GOVT, AGG, BND, VTI)
*   **Optimization Target:** max\_sharpe
*   **Maximum Position:** 15%

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


...

You're right to push for a more rigorous definition of "moderate risk." Here's a breakdown of how I approached it, and the limitations of the available tools:

**1. Defining "Moderate Risk":**

Unfortunately, the available tools don't allow for a direct input of a risk tolerance score or a utility function. Therefore, I had to infer "moderate risk" from the following:

*   **Elena's Explicit Statement:** She stated she has a moderate risk tolerance.
*   **Time Horizon:** An 18-year time horizon allows for more risk-taking than a shorter horizon.
*   **Implicit Preference:** Her interest in the maximum Sharpe ratio suggests she's not entirely risk-averse; a purely risk-averse investor might only consider minimum volatility.

Based on these factors, I interpreted "moderate risk" as a willingness to accept some volatility in exchange for potentially higher returns, but not so much volatility that it would cause undue stress or jeopardize the education fund's goals. A more precise definiti

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


The Purple Agent acknowledges the limitations of the available tools and provides a reasonable justification for its recommendation. However, I want to explore the agent's understanding of Hierarchical Risk Parity (HRP) more deeply, especially since it was presented as an alternative approach. The agent mentions HRP focuses on diversification and risk allocation, but I want to know if it considered the specific benefits of HRP in handling non-normal returns or unstable correlation structures, which are often cited as advantages of this method.

FOLLOW-UP QUESTION:

You presented Hierarchical Risk Parity (HRP) as an alternative approach. While you mentioned its focus on diversification and risk allocation, HRP is often touted for its ability to handle non-normal returns and unstable correlation structures better than traditional mean-variance optimization. Did you consider these specific advantages of HRP when evaluating it against the Maximum Sharpe Ratio portfolio, especially given th

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


You're right to press on the specific advantages of HRP. My previous answer was too general. Here's a more detailed explanation of how I considered HRP's strengths:

**Understanding HRP's Advantages:**

HRP distinguishes itself from traditional mean-variance optimization (MVO) in several key ways, particularly relevant to long-term investing:

*   **Non-Normal Returns:** MVO assumes returns are normally distributed, which is often not the case in real-world markets. HRP is less sensitive to this assumption because it doesn't rely on the mean-variance framework. It uses hierarchical clustering to group assets based on their correlation structure, and then allocates weights based on the risk contribution of each cluster.
*   **Unstable Correlation Structures:** Correlations between assets can change over time, especially during market regime shifts. MVO is highly sensitive to these changes, as it relies on a fixed covariance matrix. HRP is more robust because it uses a hierarchical struc

## 5. Key Takeaways

### A2A Protocol Pattern

The A2A (Agent-to-Agent) evaluation uses iterative communication:

```python
from agents import (
    create_a2a_purple_agent,  # Agent being evaluated
    create_a2a_green_agent,   # Evaluator agent
    run_a2a_evaluation,
)

# Create agents
purple_agent = create_a2a_purple_agent(provider=LLMProvider.GEMINI)
green_agent = create_a2a_green_agent(retriever, provider=LLMProvider.GEMINI)

# Run A2A evaluation with 5 rounds
result = run_a2a_evaluation(
    task_description="Build a conservative portfolio",
    portfolio_agent=purple_agent,
    evaluator_agent=green_agent,
    session_id="evaluation_001",
    max_rounds=5  # Control conversation length
)

# Access results
print(f"Score: {result.overall_score}/10")
print(f"Messages: {len(result.conversation)}")  # 2*max_rounds + 1
```

### Iterative Communication Flow

The protocol runs for `max_rounds` iterations:

| Round | Green Agent Action | Purple Agent Action |
|-------|-------------------|---------------------|
| 1 | Initial request | Portfolio recommendation |
| 2 | Follow-up question | Clarification/details |
| 3 | Probe deeper | Justification |
| ... | ... | ... |
| Final | Assessment (scores + feedback) | - |

### Conversation Structure

Each round produces 2 messages (Green query + Purple response):
- `max_rounds=3` → 7 messages (6 exchanges + 1 assessment)
- `max_rounds=5` → 11 messages (10 exchanges + 1 assessment)

The Green agent uses tools (RAG, web search) during follow-up rounds to gather context for evaluation.

In [11]:
# Flush Langfuse events
flush_langfuse()
print("Langfuse events flushed!")
print("\nCheck Langfuse dashboard for:")
print("  - A2A evaluation traces with Green/Purple agent communication")
print("  - Sessions: notebook3_demo, notebook3_a2a, notebook3_complex")
print("  - Hierarchical view: A2A Evaluation → Agent chains → Tool calls")

Langfuse events flushed!

Check Langfuse dashboard for:
  - A2A evaluation traces with Green/Purple agent communication
  - Sessions: notebook3_demo, notebook3_a2a, notebook3_complex
  - Hierarchical view: A2A Evaluation → Agent chains → Tool calls
